In [2]:
#Upload file
from google.colab import files
uploaded = files.upload()

Saving cleaned_news.csv to cleaned_news.csv


In [3]:
#Load df
import pandas as pd
df = pd.read_csv("cleaned_news.csv")
print(f"Total articles: {len(df)}")
print(df.head(5))

Total articles: 10069
                                               title  \
0  Komisi X Akan Minta Penjelasan Kemendikdasmen ...   
1  Full Senyum, Prabowo-Megawati Gandengan Tangan...   
2  Purbaya soal Ekspor via PT DSI: Bukan Program ...   
3  Prabowo: Ekonomi RI Tumbuh tapi Apakah Sudah D...   
4  Prabowo: Tak Ada Bangsa yang Kasihan Sama Kita...   

                                                 url        date  source  
0  https://nasional.kompas.com/read/2026/06/01/11...  2026-06-01  Kompas  
1  https://nasional.kompas.com/read/2026/06/01/11...  2026-06-01  Kompas  
2  https://money.kompas.com/read/2026/06/01/11155...  2026-06-01  Kompas  
3  https://nasional.kompas.com/read/2026/06/01/11...  2026-06-01  Kompas  
4  https://nasional.kompas.com/read/2026/06/01/10...  2026-06-01  Kompas  


In [4]:
#Install libraries
!pip install transformers torch sentencepiece -q

In [5]:
#Load IndoBERT Sentiment Model
from transformers import pipeline

# IndoBERT fine-tuned for sentiment analysis
sentiment_pipeline = pipeline(
    "text-classification",
    model="mdhugol/indonesia-bert-sentiment-classification",
    tokenizer="mdhugol/indonesia-bert-sentiment-classification"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: mdhugol/indonesia-bert-sentiment-classification
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [6]:
#Test sentiment model
test_titles = [
    "Prabowo Pimpin Upacara Hari Pancasila, Megawati Hadir",
    "Kritik Keras DPR terhadap Kebijakan Ekonomi Prabowo",
    "Prabowo Lantik Menteri Baru di Istana Negara"
]

for title in test_titles:
    result = sentiment_pipeline(title)
    print(f"{result[0]['label']} ({result[0]['score']:.3f}) → {title[:60]}")

LABEL_1 (0.922) → Prabowo Pimpin Upacara Hari Pancasila, Megawati Hadir
LABEL_1 (0.757) → Kritik Keras DPR terhadap Kebijakan Ekonomi Prabowo
LABEL_1 (0.914) → Prabowo Lantik Menteri Baru di Istana Negara


In [8]:
#Run sentiment on all articles
from tqdm import tqdm

label_map = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"}

# Ensure all titles are strings
df["title"] = df["title"].fillna("").astype(str)
df = df[df["title"].str.strip() != ""]

results = []
batch_size = 32

for i in tqdm(range(0, len(df), batch_size)):
    batch = df["title"].iloc[i:i+batch_size].tolist()
    preds = sentiment_pipeline(batch, truncation=True, max_length=128)
    for pred in preds:
        results.append({
            "label": label_map[pred["label"]],
            "score": pred["score"]
        })

df["sentiment"] = [r["label"] for r in results]
df["sentiment_score"] = [r["score"] for r in results]

print(f"Sentiment distribution:")
print(df["sentiment"].value_counts())

100%|██████████| 315/315 [01:34<00:00,  3.33it/s]

Sentiment distribution:
sentiment
neutral     9780
positive     225
negative      63
Name: count, dtype: int64


In [9]:
#Save sentiment results
df.to_csv("sentiment_results.csv", index=False)
print(df[["title", "sentiment", "sentiment_score"]].head(10))

                                               title sentiment  \
0  Komisi X Akan Minta Penjelasan Kemendikdasmen ...   neutral   
1  Full Senyum, Prabowo-Megawati Gandengan Tangan...   neutral   
2  Purbaya soal Ekspor via PT DSI: Bukan Program ...   neutral   
3  Prabowo: Ekonomi RI Tumbuh tapi Apakah Sudah D...  positive   
4  Prabowo: Tak Ada Bangsa yang Kasihan Sama Kita...   neutral   
5  Daftar Mantan Presiden-Wapres Hadiri Upacara H...   neutral   
6  Momen Prabowo Berdoa di Depan Peti Jenazah Rya...   neutral   
7  Meski Dipersilakan Prabowo, Megawati Tolak Ber...   neutral   
8  Didampingi Gibran, Prabowo Pimpin Upacara Hari...   neutral   
9  Romo Syafi’i: Gagasan Prabowo Bentuk Kemenhaj ...   neutral   

   sentiment_score  
0         0.804089  
1         0.902486  
2         0.967618  
3         0.823569  
4         0.667313  
5         0.841966  
6         0.885046  
7         0.954255  
8         0.941552  
9         0.943393  


In [10]:
from google.colab import files
files.download("sentiment_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>